# Corporacion Favorita: EDA и первые baseline-модели

Цель ноутбука:
- быстро понять структуру данных и масштаб таблиц;
- выделить основные временные паттерны, промо-эффекты и внешние драйверы;
- зафиксировать риски для baseline-модели и проверить пару простых holdout-baseline.

Подход:
- небольшие таблицы читаем целиком;
- `train.csv` сканируем чанками и складываем агрегаты в `.cache/favorita/`, чтобы повторный запуск был быстрым.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from favorita_eda_utils import (
    build_daily_external_frame,
    build_test_coverage,
    build_train_eda_bundle,
    dataset_catalog,
    load_reference_tables,
)
from favorita_baselines import (
    build_baseline_validation_artifacts,
    predict_hierarchical_baseline,
    predict_recent_mean_baseline,
    summarize_baseline_results,
    weighted_rmsle,
)
from favorita_models import run_lightgbm_validation_experiment
from favorita_models import (
    run_single_fold_time_series_experiment,
    run_time_series_cross_validation,
    train_final_time_series_model,
)
from favorita_catboost import (
    run_single_fold_catboost_experiment,
    run_catboost_optuna_search,
    train_final_catboost_model,
)
from favorita_tft import (
    DEFAULT_TFT_DATASET_CONFIG,
    DEFAULT_TFT_PARAMS,
    DEFAULT_TRAINER_PARAMS,
    _build_prediction_frame,
    _build_tft_datasets,
    _prepare_tft_fold_frames,
    _prepare_tft_test_frames,
    _predict_tft_horizon,
)

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA_DIR = Path(".")


In [ ]:
catalog = dataset_catalog(DATA_DIR)
tables = load_reference_tables(DATA_DIR)

stores = tables["stores"]
items = tables["items"]
transactions = tables["transactions"]
oil = tables["oil"]
holidays = tables["holidays"]
test = tables["test"]

reference_overview = pd.DataFrame(
    [
        {
            "table": "stores",
            "rows": len(stores),
            "cols": stores.shape[1],
            "date_min": pd.NaT,
            "date_max": pd.NaT,
            "missing_share": stores.isna().mean().mean(),
        },
        {
            "table": "items",
            "rows": len(items),
            "cols": items.shape[1],
            "date_min": pd.NaT,
            "date_max": pd.NaT,
            "missing_share": items.isna().mean().mean(),
        },
        {
            "table": "transactions",
            "rows": len(transactions),
            "cols": transactions.shape[1],
            "date_min": transactions["date"].min(),
            "date_max": transactions["date"].max(),
            "missing_share": transactions.isna().mean().mean(),
        },
        {
            "table": "oil",
            "rows": len(oil),
            "cols": oil.shape[1],
            "date_min": oil["date"].min(),
            "date_max": oil["date"].max(),
            "missing_share": oil.isna().mean().mean(),
        },
        {
            "table": "holidays_events",
            "rows": len(holidays),
            "cols": holidays.shape[1],
            "date_min": holidays["date"].min(),
            "date_max": holidays["date"].max(),
            "missing_share": holidays.isna().mean().mean(),
        },
        {
            "table": "test",
            "rows": len(test),
            "cols": test.shape[1],
            "date_min": test["date"].min(),
            "date_max": test["date"].max(),
            "missing_share": test.isna().mean().mean(),
        },
    ]
)

display(catalog)
display(reference_overview)

## Метаданные по магазинам и товарам

До чтения `train.csv` полезно понять, насколько неоднородны сами сущности:
- в `stores.csv` есть тип магазина, география и cluster;
- в `items.csv` есть family/class/perishable, что сразу даёт естественные агрегирующие признаки.

In [ ]:
store_type_counts = stores["type"].value_counts().sort_index().rename_axis("type").reset_index(name="stores")
top_families_by_items = (
    items["family"].value_counts().head(12).sort_values().rename_axis("family").reset_index(name="item_count")
)
perishable_share = pd.DataFrame(
    {
        "bucket": ["non-perishable", "perishable"],
        "share": [1 - items["perishable"].mean(), items["perishable"].mean()],
    }
)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.barplot(data=store_type_counts, x="type", y="stores", ax=axes[0], color="#457b9d")
axes[0].set_title("Stores by type")
axes[0].set_xlabel("Store type")
axes[0].set_ylabel("Number of stores")

sns.barplot(data=top_families_by_items, x="item_count", y="family", ax=axes[1], color="#2a9d8f")
axes[1].set_title("Top item families by SKU count")
axes[1].set_xlabel("Number of items")
axes[1].set_ylabel("")

sns.barplot(data=perishable_share, x="bucket", y="share", ax=axes[2], color="#e76f51")
axes[2].set_title("Perishable share in item catalog")
axes[2].set_xlabel("")
axes[2].set_ylabel("Share")
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Full-train summary через чанки

`train.csv` содержит более 125 млн строк, поэтому ниже используется один полный проход по данным:
- считаются глобальные статистики;
- собираются ежедневные агрегаты;
- отдельно агрегируются продажи по `family` и `store_nbr`;
- результаты кешируются, чтобы повторный запуск ноутбука не перечитывал весь файл заново.

In [ ]:
train_eda = build_train_eda_bundle(DATA_DIR)

daily_sales = train_eda["daily_sales"].copy()
family_sales = train_eda["family_sales"].copy()
store_sales = train_eda["store_sales"].copy()
promotion_summary = train_eda["promotion_summary"].copy()
daily_frame = build_daily_external_frame(daily_sales, transactions, oil)

overview = train_eda["overview"]
train_summary = pd.DataFrame(
    [
        {"metric": "train_rows", "value": f"{overview['rows']:,}"},
        {"metric": "train_start", "value": str(overview["date_min"].date())},
        {"metric": "train_end", "value": str(overview["date_max"].date())},
        {"metric": "n_days", "value": f"{overview['n_days']:,}"},
        {"metric": "unique_stores", "value": f"{overview['unique_stores']:,}"},
        {"metric": "unique_items", "value": f"{overview['unique_items']:,}"},
        {"metric": "missing_onpromotion_rate", "value": f"{overview['missing_onpromotion_rate']:.2%}"},
        {"metric": "negative_sales_rate", "value": f"{overview['negative_sales_rate']:.4%}"},
        {"metric": "total_unit_sales", "value": f"{overview['total_unit_sales']:,.0f}"},
    ]
)

promotion_display = promotion_summary.copy()
promotion_display["records"] = promotion_display["records"].map(lambda value: f"{value:,}")
promotion_display["record_share_known"] = promotion_display["record_share_known"].map(lambda value: f"{value:.2%}")
promotion_display["unit_sales"] = promotion_display["unit_sales"].map(lambda value: f"{value:,.0f}")
promotion_display["avg_unit_sales_per_record"] = promotion_summary["avg_unit_sales_per_record"].round(3)

display(train_summary)
display(build_test_coverage(test, train_eda["train_items"], train_eda["train_stores"]))
display(promotion_display)

Важно: промо-эффект здесь интерпретируется аккуратно.
В `train.csv` отсутствуют пары `date/store/item` с нулевыми продажами, поэтому сравнение `onpromotion=True` и `False`
показывает разницу только на наблюдаемых продажах, а не полноценный uplift относительно всей полки.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

daily_frame["unit_sales"].plot(ax=ax, alpha=0.35, linewidth=1, label="daily unit_sales")
daily_frame["rolling_28d"].plot(ax=ax, linewidth=2.5, label="28-day rolling mean")

ax.axvline(pd.Timestamp("2016-04-16"), color="#d62828", linestyle="--", linewidth=2, label="earthquake")
ax.axvline(test["date"].min(), color="#264653", linestyle="--", linewidth=2, label="test start")
ax.set_title("Daily sales across the full training horizon")
ax.set_xlabel("")
ax.set_ylabel("Unit sales")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_profile = daily_frame.groupby("weekday", as_index=False)["unit_sales"].mean()
weekday_profile["weekday"] = pd.Categorical(weekday_profile["weekday"], categories=weekday_order, ordered=True)
weekday_profile = weekday_profile.sort_values("weekday")

payday_profile = daily_frame.groupby("is_payday", as_index=False)["unit_sales"].mean()
payday_profile["bucket"] = payday_profile["is_payday"].map(
    {False: "regular day", True: "15th or month-end"}
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.barplot(data=weekday_profile, x="weekday", y="unit_sales", ax=axes[0], color="#2a9d8f")
axes[0].set_title("Average daily sales by weekday")
axes[0].set_xlabel("")
axes[0].set_ylabel("Average unit sales")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=payday_profile, x="bucket", y="unit_sales", ax=axes[1], color="#e76f51")
axes[1].set_title("Payday effect")
axes[1].set_xlabel("")
axes[1].set_ylabel("Average unit sales")

plt.tight_layout()
plt.show()

In [ ]:
corr = daily_frame[["unit_sales", "transactions", "dcoilwtico"]].corr()
display(corr)

normalized = daily_frame[["unit_sales", "dcoilwtico"]].dropna().apply(
    lambda column: (column - column.mean()) / column.std()
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.scatterplot(
    data=daily_frame,
    x="transactions",
    y="unit_sales",
    alpha=0.45,
    s=40,
    ax=axes[0],
    color="#457b9d",
)
axes[0].set_title("Sales vs. transactions")
axes[0].set_xlabel("Daily transactions")
axes[0].set_ylabel("Daily unit sales")

normalized.plot(ax=axes[1], linewidth=2)
axes[1].set_title("Normalized sales vs. oil price")
axes[1].set_xlabel("")
axes[1].set_ylabel("z-score")

plt.tight_layout()
plt.show()

In [ ]:
top_families = family_sales.nlargest(12, "unit_sales").sort_values("unit_sales")
top_families["unit_sales_mn"] = top_families["unit_sales"] / 1_000_000

store_plot = store_sales.copy()
store_plot["unit_sales_mn"] = store_plot["unit_sales"] / 1_000_000

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.barplot(data=top_families, x="unit_sales_mn", y="family", ax=axes[0], color="#1d3557")
axes[0].set_title("Top families by total sales")
axes[0].set_xlabel("Total unit sales, millions")
axes[0].set_ylabel("")

sns.boxplot(data=store_plot, x="type", y="unit_sales_mn", ax=axes[1], color="#8ecae6")
axes[1].set_title("Store sales dispersion by store type")
axes[1].set_xlabel("Store type")
axes[1].set_ylabel("Total unit sales, millions")

plt.tight_layout()
plt.show()

display(
    store_sales.head(10)[
        ["store_nbr", "city", "state", "type", "cluster", "unit_sales", "sales_share"]
    ]
)

In [ ]:
holiday_pivot = holidays.groupby(["type", "locale"]).size().unstack(fill_value=0)
transferred_examples = holidays.loc[
    holidays["transferred"],
    ["date", "type", "locale", "locale_name", "description", "transferred"],
].head(10)

earthquake_window = daily_frame.loc["2016-03-15":"2016-05-31", ["unit_sales"]]

display(holiday_pivot)
display(transferred_examples)

fig, ax = plt.subplots(figsize=(16, 5))
earthquake_window["unit_sales"].plot(ax=ax, linewidth=2, color="#d62828")
ax.axvline(pd.Timestamp("2016-04-16"), color="black", linestyle="--", linewidth=2)
ax.set_title("Sales around the April 16, 2016 earthquake")
ax.set_xlabel("")
ax.set_ylabel("Unit sales")

plt.tight_layout()
plt.show()

## Holdout validation и baseline-модели

Ниже используем простую time-based валидацию:
- validation horizon = последние 16 дней train, то есть с `2017-07-31` по `2017-08-15`;
- fit window = предыдущие 112 дней;
- метрика = weighted RMSLE с весами `1.25` для perishable и `1.0` иначе.

Ограничение валидации:
- в `train.csv` отсутствуют нулевые продажи, поэтому holdout ниже считается только на наблюдаемых строках;
- это полезный относительный sanity check для baseline-моделей, но не идеальная реплика Kaggle leaderboard.

In [ ]:
baseline_artifacts = build_baseline_validation_artifacts(DATA_DIR)
baseline_meta = pd.DataFrame(
    [
        {"metric": "fit_start", "value": str(baseline_artifacts["metadata"]["fit_start"].date())},
        {"metric": "valid_start", "value": str(baseline_artifacts["metadata"]["valid_start"].date())},
        {"metric": "valid_end", "value": str(baseline_artifacts["metadata"]["valid_end"].date())},
        {"metric": "fit_rows", "value": f"{baseline_artifacts['metadata']['fit_rows']:,}"},
        {"metric": "valid_rows_observed_only", "value": f"{baseline_artifacts['metadata']['valid_rows']:,}"},
        {"metric": "global_mean_target", "value": f"{baseline_artifacts['metadata']['global_mean']:.3f}"},
    ]
)
display(baseline_meta)

In [ ]:
baseline_recent_mean = predict_recent_mean_baseline(baseline_artifacts)
baseline_hierarchical = predict_hierarchical_baseline(baseline_artifacts)

baseline_scores = summarize_baseline_results(
    {
        "recent_mean_28d": baseline_recent_mean,
        "hierarchical_weekday_promo": baseline_hierarchical,
    }
)
display(baseline_scores)

source_mix = (
    baseline_hierarchical["source"]
    .value_counts(normalize=True)
    .rename("row_share")
    .reset_index()
    .rename(columns={"index": "source"})
)
display(source_mix)

In [ ]:
baseline_daily = (
    baseline_hierarchical.groupby("date", as_index=False)[["actual", "prediction"]]
    .sum()
    .rename(columns={"prediction": "hierarchical_prediction"})
)
baseline_daily = baseline_daily.merge(
    baseline_recent_mean.groupby("date", as_index=False)["prediction"].sum(),
    on="date",
    how="left",
).rename(columns={"prediction": "recent_mean_prediction"})

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

baseline_daily.plot(
    x="date",
    y=["actual", "recent_mean_prediction", "hierarchical_prediction"],
    ax=axes[0],
    linewidth=2,
)
axes[0].set_title("Daily totals on validation horizon")
axes[0].set_xlabel("")
axes[0].set_ylabel("Observed-row unit sales")

sns.scatterplot(
    data=baseline_hierarchical.sample(min(100_000, len(baseline_hierarchical)), random_state=42),
    x="actual",
    y="prediction",
    alpha=0.25,
    s=20,
    ax=axes[1],
    color="#1d3557",
)
axes[1].set_title("Hierarchical baseline: row-level fit")
axes[1].set_xlabel("Actual sales")
axes[1].set_ylabel("Prediction")

plt.tight_layout()
plt.show()

## Первая обучаемая модель: feature-based LightGBM

После lookup-baseline следующий логичный шаг:
- взять тот же holdout;
- собрать табличные признаки по календарю, promotions, metadata, holidays, oil и transactions;
- добавить исторические средние по `store-item` и `family`;
- обучить первую компактную `LightGBM`-модель на `log1p(unit_sales)`.

Ограничение этого шага:
- `transactions` доступны только на train-периоде;
- значит этот блок пока решает задачу validation/prototyping, а не готовый Kaggle inference pipeline.

In [ ]:
lgbm_result = run_lightgbm_validation_experiment(DATA_DIR)

lgbm_meta = pd.DataFrame(
    [
        {"metric": "fit_start", "value": str(lgbm_result["metadata"]["fit_start"].date())},
        {"metric": "valid_start", "value": str(lgbm_result["metadata"]["valid_start"].date())},
        {"metric": "valid_end", "value": str(lgbm_result["metadata"]["valid_end"].date())},
        {"metric": "fit_rows", "value": f"{lgbm_result['metadata']['fit_rows']:,}"},
        {"metric": "valid_rows", "value": f"{lgbm_result['metadata']['valid_rows']:,}"},
        {"metric": "n_features", "value": len(lgbm_result["feature_importance"])},
    ]
)

display(lgbm_meta)
display(lgbm_result["scores"])
display(lgbm_result["feature_importance"].head(12))

In [ ]:
lgbm_daily = lgbm_result["daily_validation"].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

lgbm_daily.plot(
    x="date",
    y=["actual", "hierarchical_prediction", "lightgbm_prediction"],
    ax=axes[0],
    linewidth=2,
)
axes[0].set_title("Validation horizon: baseline vs LightGBM")
axes[0].set_xlabel("")
axes[0].set_ylabel("Observed-row unit sales")

top_gain = lgbm_result["feature_importance"].head(12).sort_values("importance_gain")
sns.barplot(data=top_gain, x="importance_gain", y="feature", ax=axes[1], color="#457b9d")
axes[1].set_title("Top LightGBM features by gain")
axes[1].set_xlabel("Gain importance")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## Полноценная стратегия для временных рядов: rolling-origin CV

Для реального обучения одной последней валидации мало, поэтому ниже используется более строгая схема:
- horizon = 16 дней, как в `test.csv`;
- 4 rolling-origin fold'а с шагом 28 дней;
- grid по lookback-окнам `112` и `224` дней;
- основной pipeline строится **без `transactions`**, чтобы модель была применима на test-time;
- вся доступная история train дополнительно используется через full-history aggregate priors.

Важная инженерная оговорка:
- скармливать все 125M observed rows в один raw fit здесь хуже и по памяти, и по статистике;
- для нестационарного retail-ряда разумнее подобрать рабочее окно истории через rolling CV, а старую историю использовать как priors/backoff-признаки.

In [ ]:
tscv_result = run_time_series_cross_validation(
    DATA_DIR,
    lookback_grid=(112, 224),
    horizon_days=16,
    step_days=28,
    n_folds=4,
    include_transactions=False,
)

tscv_meta = pd.DataFrame(
    [
        {"metric": "best_lookback_days", "value": int(tscv_result["metadata"]["best_lookback_days"])},
        {"metric": "best_mean_wrmsle", "value": f"{tscv_result['metadata']['best_mean_score']:.6f}"},
        {"metric": "n_folds", "value": tscv_result["metadata"]["n_folds"]},
        {"metric": "horizon_days", "value": tscv_result["metadata"]["horizon_days"]},
        {"metric": "step_days", "value": tscv_result["metadata"]["step_days"]},
    ]
)

display(tscv_meta)
display(tscv_result["summary"])
display(tscv_result["fold_scores"])
display(tscv_result["feature_importance_summary"].head(15))

In [ ]:
cv_plot = tscv_result["fold_scores"].copy()
cv_plot["valid_start"] = cv_plot["valid_start"].dt.strftime("%Y-%m-%d")

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.lineplot(
    data=cv_plot,
    x="valid_start",
    y="weighted_rmsle",
    hue="lookback_days",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Rolling CV by validation fold")
axes[0].set_xlabel("Validation start")
axes[0].set_ylabel("Weighted RMSLE")
axes[0].tick_params(axis="x", rotation=30)

top_cv_gain = tscv_result["feature_importance_summary"].head(12).sort_values("importance_gain")
sns.barplot(data=top_cv_gain, x="importance_gain", y="feature", ax=axes[1], color="#264653")
axes[1].set_title("Top CV features by mean gain")
axes[1].set_xlabel("Mean gain importance")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## Финальное обучение и inference на `test.csv`

После CV модель обучается заново на train до `2017-08-15` с лучшим окном истории и тем же test-time-safe feature set.
Результат сохраняется сразу в Kaggle-ready submission.

In [ ]:
final_model_result = train_final_time_series_model(
    DATA_DIR,
    lookback_days=tscv_result["metadata"]["best_lookback_days"],
    include_transactions=False,
)

final_meta = pd.DataFrame(
    [
        {"metric": "fit_start", "value": str(final_model_result["metadata"]["fit_start"].date())},
        {"metric": "train_end", "value": str(final_model_result["metadata"]["train_end"].date())},
        {"metric": "test_start", "value": str(final_model_result["metadata"]["test_start"].date())},
        {"metric": "test_end", "value": str(final_model_result["metadata"]["test_end"].date())},
        {"metric": "fit_rows", "value": f"{final_model_result['metadata']['fit_rows']:,}"},
        {"metric": "submission_path", "value": final_model_result["metadata"]["submission_path"]},
    ]
)

display(final_meta)
display(final_model_result["feature_importance"].head(15))
display(final_model_result["submission_head"])

In [ ]:
latest_fold_start = tscv_result["folds"]["valid_start"].max()
latest_fold_result = run_single_fold_time_series_experiment(
    valid_start=latest_fold_start,
    lookback_days=tscv_result["metadata"]["best_lookback_days"],
    include_transactions=False,
)

submission_frame = pd.read_csv(final_model_result["metadata"]["submission_path"], compression="gzip")
test_submission_index = pd.read_csv(DATA_DIR / "test.csv", usecols=["id", "date"], parse_dates=["date"])
forecast_daily = (
    test_submission_index
    .merge(submission_frame, on="id", how="left")
    .groupby("date", as_index=False)["unit_sales"]
    .sum()
    .rename(columns={"unit_sales": "forecast_unit_sales"})
)

history_cutoff = pd.Timestamp(final_model_result["metadata"]["train_end"]) - pd.Timedelta(days=90)
history_window = daily_sales.reset_index()[["date", "unit_sales"]].rename(
    columns={"unit_sales": "actual_unit_sales"}
)
history_window = history_window[history_window["date"] >= history_cutoff].copy()
validation_daily = latest_fold_result["daily_validation"].copy()
validation_end = validation_daily["date"].max()
test_start = pd.Timestamp(final_model_result["metadata"]["test_start"])
test_end = pd.Timestamp(final_model_result["metadata"]["test_end"])

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

validation_daily.plot(
    x="date",
    y=["actual", "prediction"],
    ax=axes[0],
    linewidth=2.5,
)
axes[0].set_title(
    f"Validation fold inside train: {latest_fold_start.date()} to {validation_end.date()}"
)
axes[0].set_xlabel("")
axes[0].set_ylabel("Daily aggregated unit sales")
axes[0].legend(title="Ground truth available")

history_window.plot(x="date", y="actual_unit_sales", ax=axes[1], linewidth=2.5, label="train actual")
forecast_daily.plot(x="date", y="forecast_unit_sales", ax=axes[1], linewidth=2.5, label="test forecast")
axes[1].axvline(
    test_start,
    color="#d62828",
    linestyle="--",
    linewidth=2,
)
axes[1].set_title(
    f"Real Kaggle test: {test_start.date()} to {test_end.date()} (no actuals released)"
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Daily aggregated unit sales")
axes[1].legend(title="Test actuals unavailable")

plt.tight_layout()
plt.show()

## Текущая боевая стратегия: CatBoost + pseudo-public validation

После загрузки `LightGBM` submission на Kaggle стало ясно, что observed-row holdout слишком оптимистичен.
Поэтому ниже:

- используем full-panel validation на реальных `store-item` парах из `test.csv`;
- считаем агрегаты по полному observed fit;
- сам `CatBoost` обучаем на sampled recent train;
- отдельно держим `Optuna`, но короткий поиск пока служит скорее infrastructure check, чем финальным тюнингом.

In [ ]:
catboost_params = {
    "iterations": 120,
    "depth": 8,
    "learning_rate": 0.05,
    "border_count": 128,
    "verbose": False,
}
catboost_postprocess = {
    "history_scale": 3.0,
    "min_model_weight": 1.0,
    "unseen_model_weight": 1.0,
}

catboost_single_fold = run_single_fold_catboost_experiment(
    valid_start=latest_fold_start,
    lookback_days=224,
    fit_max_rows=300_000,
    eval_max_rows=120_000,
    zero_sample_size=80_000,
    zero_sample_days=28,
    model_params=catboost_params,
    history_scale=catboost_postprocess["history_scale"],
    min_model_weight=catboost_postprocess["min_model_weight"],
    unseen_model_weight=catboost_postprocess["unseen_model_weight"],
)

catboost_meta = pd.DataFrame(
    [
        {"metric": "fit_start", "value": str(catboost_single_fold["metadata"]["fit_start"].date())},
        {"metric": "valid_start", "value": str(catboost_single_fold["metadata"]["valid_start"].date())},
        {"metric": "valid_end", "value": str(catboost_single_fold["metadata"]["valid_end"].date())},
        {"metric": "fit_observed_rows", "value": f"{catboost_single_fold['metadata']['fit_observed_rows']:,}"},
        {"metric": "fit_train_rows", "value": f"{catboost_single_fold['metadata']['fit_train_rows']:,}"},
        {"metric": "valid_rows_full_panel", "value": f"{catboost_single_fold['metadata']['valid_rows']:,}"},
        {"metric": "raw_catboost_wrmsle", "value": f"{catboost_single_fold['raw_model_score']:.6f}"},
        {"metric": "final_score_with_current_postprocess", "value": f"{catboost_single_fold['score']:.6f}"},
    ]
)

display(catboost_meta)
display(catboost_single_fold["feature_importance"].head(15))

In [ ]:
catboost_daily = catboost_single_fold["daily_validation"].copy()

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

catboost_daily.plot(
    x="date",
    y=["actual", "catboost_raw_prediction"],
    ax=axes[0],
    linewidth=2.5,
)
axes[0].set_title("CatBoost on pseudo-public validation fold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Daily aggregated unit sales")

top_cb_gain = catboost_single_fold["feature_importance"].head(12).sort_values("importance")
sns.barplot(data=top_cb_gain, x="importance", y="feature", ax=axes[1], color="#2a9d8f")
axes[1].set_title("Top CatBoost features")
axes[1].set_xlabel("Feature importance")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
catboost_optuna = run_catboost_optuna_search(
    lookback_days=224,
    fit_max_rows=300_000,
    eval_max_rows=120_000,
    zero_sample_size=80_000,
    zero_sample_days=28,
    n_trials=2,
)

display(pd.DataFrame([catboost_optuna["best_postprocess"]]))
display(pd.DataFrame([catboost_optuna["best_model_params"]]))
display(catboost_optuna["trials"])

In [ ]:
catboost_final = train_final_catboost_model(
    lookback_days=224,
    fit_max_rows=300_000,
    zero_sample_size=80_000,
    zero_sample_days=28,
    model_params=catboost_params,
    postprocess_params=catboost_postprocess,
)

catboost_final_meta = pd.DataFrame(
    [
        {"metric": "fit_start", "value": str(catboost_final["metadata"]["fit_start"].date())},
        {"metric": "train_end", "value": str(catboost_final["metadata"]["train_end"].date())},
        {"metric": "test_start", "value": str(catboost_final["metadata"]["test_start"].date())},
        {"metric": "test_end", "value": str(catboost_final["metadata"]["test_end"].date())},
        {"metric": "fit_observed_rows", "value": f"{catboost_final['metadata']['fit_observed_rows']:,}"},
        {"metric": "fit_train_rows", "value": f"{catboost_final['metadata']['fit_train_rows']:,}"},
        {"metric": "submission_path", "value": catboost_final["metadata"]["submission_path"]},
    ]
)

display(catboost_final_meta)
display(catboost_final["feature_importance"].head(15))
display(catboost_final["submission_head"])
display(catboost_final["unseen_summary"])

## TFT baseline (PyTorch Forecasting, pretrained inference)

? ???? ????? ???????? TFT ?????????.
???????????? ????????????? `.pt` checkpoint ? ??????????? ?????? inference:
- pseudo-public validation ?? 16 ????;
- ????????? test-time inference + submission.


In [ ]:
tft_params = DEFAULT_TFT_PARAMS.copy()
tft_trainer_params = DEFAULT_TRAINER_PARAMS.copy()

tft_dataset_params = {
    "lookback_days": 224,
    "horizon_days": 16,
    "max_encoder_length": 56,
    "max_prediction_length": 16,
    "max_series": 15_000,
    "min_history_points": 28,
}

# Put your saved checkpoint here.
tft_checkpoint_candidates = [
    DATA_DIR / "artifacts" / "favorita_tft_state_dict.pt",
    DATA_DIR / "favorita_tft_state_dict.pt",
]


def _resolve_tft_checkpoint_path(candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "TFT checkpoint not found. Checked: " + ", ".join(str(path) for path in candidates)
    )


def _load_tft_from_pt(train_dataset, checkpoint_path: Path, params: dict):
    import torch
    from pytorch_forecasting import TemporalFusionTransformer
    from pytorch_forecasting.metrics import RMSE

    model = TemporalFusionTransformer.from_dataset(
        train_dataset,
        learning_rate=float(params["learning_rate"]),
        hidden_size=int(params["hidden_size"]),
        attention_head_size=int(params["attention_head_size"]),
        hidden_continuous_size=int(params["hidden_continuous_size"]),
        dropout=float(params["dropout"]),
        loss=RMSE(),
        log_interval=-1,
    )

    payload = torch.load(checkpoint_path, map_location="cpu")
    state_dict = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"[warn] Missing keys: {len(missing)}")
    if unexpected:
        print(f"[warn] Unexpected keys: {len(unexpected)}")
    model.eval()
    return model


try:
    tft_checkpoint_path = _resolve_tft_checkpoint_path(tft_checkpoint_candidates)
    print("Using TFT checkpoint:", tft_checkpoint_path)

    dataset_cfg = DEFAULT_TFT_DATASET_CONFIG.copy()
    dataset_cfg.update(
        {
            "max_encoder_length": tft_dataset_params["max_encoder_length"],
            "max_prediction_length": tft_dataset_params["max_prediction_length"],
            "max_series": tft_dataset_params["max_series"],
            "min_history_points": tft_dataset_params["min_history_points"],
            "series_sample_seed": int(tft_trainer_params["random_seed"]),
            "num_workers": int(tft_trainer_params.get("num_workers", DEFAULT_TRAINER_PARAMS["num_workers"])),
        }
    )

    fit_frame, valid_frame, _eligible_pairs, metadata = _prepare_tft_fold_frames(
        valid_start=latest_fold_start,
        lookback_days=tft_dataset_params["lookback_days"],
        horizon_days=tft_dataset_params["horizon_days"],
        max_encoder_length=tft_dataset_params["max_encoder_length"],
        min_history_points=tft_dataset_params["min_history_points"],
        max_series=tft_dataset_params["max_series"],
        random_seed=int(dataset_cfg["series_sample_seed"]),
        data_dir=DATA_DIR,
        use_cache=True,
        force=False,
    )

    valid_eligible = valid_frame.loc[valid_frame["is_tft_eligible"].eq(1)].copy()
    tft_prediction = None
    model_summary = {"status": "skipped_no_eligible_series"}

    if not fit_frame.empty and not valid_eligible.empty:
        dataset_bundle = _build_tft_datasets(
            fit_frame=fit_frame,
            horizon_frame=valid_eligible,
            dataset_config=dataset_cfg,
        )
        model = _load_tft_from_pt(
            train_dataset=dataset_bundle["train_dataset"],
            checkpoint_path=tft_checkpoint_path,
            params=tft_params,
        )
        tft_prediction = _predict_tft_horizon(
            model=model,
            predict_loader=dataset_bundle["predict_loader"],
            horizon_frame=dataset_bundle["horizon_frame"],
            prediction_index=dataset_bundle["prediction_index"],
        )
        model_summary = {
            "status": "loaded_from_checkpoint",
            "checkpoint_path": str(tft_checkpoint_path),
        }

    valid_prediction_frame = _build_prediction_frame(valid_frame, tft_prediction)
    valid_prediction_frame["weight"] = np.where(valid_prediction_frame["perishable"].eq(1), 1.25, 1.0).astype("float32")

    score = weighted_rmsle(
        y_true=valid_prediction_frame["target"],
        y_pred=valid_prediction_frame["final_prediction"],
        weights=valid_prediction_frame["weight"],
    )

    daily_validation = (
        valid_prediction_frame.groupby("date", as_index=False)[
            ["target", "tft_raw_prediction", "fallback_prediction", "final_prediction"]
        ]
        .sum()
        .rename(columns={"target": "actual"})
        .sort_values("date", ignore_index=True)
    )

    eligibility_summary = pd.DataFrame(
        [
            {"metric": "total_pairs", "value": int(metadata["total_pairs"])} ,
            {"metric": "eligible_pairs", "value": int(metadata["eligible_pairs"])},
            {"metric": "fallback_pairs", "value": int(metadata["fallback_pairs"])},
            {"metric": "unseen_item_pairs", "value": int(metadata["unseen_item_pairs"])},
            {"metric": "insufficient_history_pairs", "value": int(metadata["insufficient_history_pairs"])},
            {"metric": "sampled_out_pairs", "value": int(metadata["sampled_out_pairs"])},
            {"metric": "eligible_rows", "value": int(valid_prediction_frame["is_tft_eligible"].sum())},
            {"metric": "rows_used_fallback", "value": int(valid_prediction_frame["used_fallback"].sum())},
        ]
    )

    valid_predictions = valid_prediction_frame[
        [
            "date",
            "store_nbr",
            "item_nbr",
            "target",
            "perishable",
            "onpromotion",
            "weight",
            "history_points",
            "is_tft_eligible",
            "sampled_out_flag",
            "unseen_item_flag",
            "tft_raw_prediction",
            "fallback_prediction",
            "fallback_source",
            "final_prediction",
            "used_fallback",
        ]
    ].copy()
    valid_predictions = valid_predictions.rename(columns={"target": "actual"})

    tft_single_fold = {
        "metadata": metadata
        | {
            "model_params": tft_params,
            "trainer_params": tft_trainer_params,
            "dataset_config": dataset_cfg,
        },
        "score": score,
        "daily_validation": daily_validation,
        "valid_predictions": valid_predictions,
        "eligibility_summary": eligibility_summary,
        "model_summary": model_summary,
    }

    tft_meta = pd.DataFrame(
        [
            {"metric": "fit_start", "value": str(tft_single_fold["metadata"]["fit_start"].date())},
            {"metric": "valid_start", "value": str(tft_single_fold["metadata"]["valid_start"].date())},
            {"metric": "valid_end", "value": str(tft_single_fold["metadata"]["valid_end"].date())},
            {"metric": "score_weighted_rmsle", "value": f"{tft_single_fold['score']:.6f}"},
            {"metric": "eligible_pairs", "value": int(tft_single_fold["metadata"]["eligible_pairs"])},
            {"metric": "fallback_pairs", "value": int(tft_single_fold["metadata"]["fallback_pairs"])},
        ]
    )
    display(tft_meta)
    display(tft_single_fold["eligibility_summary"])
    display(pd.DataFrame([tft_single_fold["model_summary"]]))
except ImportError as exc:
    print(exc)
except FileNotFoundError as exc:
    print(exc)


In [ ]:
if "tft_single_fold" in locals():
    tft_daily = tft_single_fold["daily_validation"].copy()
    fallback_share = (
        tft_single_fold["valid_predictions"]
        .groupby("date", as_index=False)["used_fallback"]
        .mean()
        .rename(columns={"used_fallback": "fallback_share"})
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))

    tft_daily.plot(
        x="date",
        y=["actual", "final_prediction", "fallback_prediction"],
        ax=axes[0],
        linewidth=2.2,
    )
    axes[0].set_title("TFT pseudo-public validation")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Daily aggregated unit sales")

    fallback_share.plot(
        x="date",
        y="fallback_share",
        ax=axes[1],
        linewidth=2.2,
        color="#d62828",
        legend=False,
    )
    axes[1].set_title("Fallback share by validation day")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Share of rows routed to fallback")
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.show()

In [ ]:
try:
    tft_checkpoint_path = _resolve_tft_checkpoint_path(tft_checkpoint_candidates)

    final_dataset_cfg = DEFAULT_TFT_DATASET_CONFIG.copy()
    final_dataset_cfg.update(
        {
            "max_encoder_length": 56,
            "max_prediction_length": 16,
            "max_series": 30_000,
            "min_history_points": 28,
            "series_sample_seed": int(tft_trainer_params["random_seed"]),
            "num_workers": int(tft_trainer_params.get("num_workers", DEFAULT_TRAINER_PARAMS["num_workers"])),
        }
    )

    fit_frame, test_frame, _eligible_pairs, metadata = _prepare_tft_test_frames(
        lookback_days=224,
        max_encoder_length=56,
        min_history_points=28,
        max_series=30_000,
        random_seed=int(final_dataset_cfg["series_sample_seed"]),
        data_dir=DATA_DIR,
        use_cache=True,
        force=False,
    )

    test_eligible = test_frame.loc[test_frame["is_tft_eligible"].eq(1)].copy()
    tft_prediction = None
    model_summary = {"status": "skipped_no_eligible_series"}

    if not fit_frame.empty and not test_eligible.empty:
        dataset_bundle = _build_tft_datasets(
            fit_frame=fit_frame,
            horizon_frame=test_eligible,
            dataset_config=final_dataset_cfg,
        )
        model = _load_tft_from_pt(
            train_dataset=dataset_bundle["train_dataset"],
            checkpoint_path=tft_checkpoint_path,
            params=tft_params,
        )
        tft_prediction = _predict_tft_horizon(
            model=model,
            predict_loader=dataset_bundle["predict_loader"],
            horizon_frame=dataset_bundle["horizon_frame"],
            prediction_index=dataset_bundle["prediction_index"],
        )
        model_summary = {
            "status": "loaded_from_checkpoint",
            "checkpoint_path": str(tft_checkpoint_path),
        }

    final_frame = _build_prediction_frame(test_frame, tft_prediction)
    submission = final_frame[["id", "final_prediction"]].rename(columns={"final_prediction": "unit_sales"}).copy()
    submission["unit_sales"] = submission["unit_sales"].astype("float32")

    submission_path = DATA_DIR / "submission_tft_pretrained_lb224_enc56.csv.gz"
    submission.to_csv(submission_path, index=False, compression="gzip")

    prediction_summary = pd.DataFrame(
        [
            {"metric": "total_test_rows", "value": int(len(final_frame))},
            {"metric": "eligible_test_rows", "value": int(final_frame["is_tft_eligible"].sum())},
            {"metric": "rows_used_fallback", "value": int(final_frame["used_fallback"].sum())},
            {"metric": "unseen_item_rows", "value": int(final_frame["unseen_item_flag"].sum())},
            {"metric": "mean_final_prediction", "value": float(final_frame["final_prediction"].mean())},
        ]
    )

    tft_final = {
        "metadata": metadata
        | {
            "model_params": tft_params,
            "trainer_params": tft_trainer_params,
            "dataset_config": final_dataset_cfg,
            "model_summary": model_summary,
            "submission_path": str(submission_path),
        },
        "submission_head": submission.head(10),
        "prediction_summary": prediction_summary,
        "submission_path": str(submission_path),
    }

    tft_final_meta = pd.DataFrame(
        [
            {"metric": "fit_start", "value": str(tft_final["metadata"]["fit_start"].date())},
            {"metric": "train_end", "value": str(tft_final["metadata"]["train_end"].date())},
            {"metric": "test_start", "value": str(tft_final["metadata"]["test_start"].date())},
            {"metric": "test_end", "value": str(tft_final["metadata"]["test_end"].date())},
            {"metric": "submission_path", "value": tft_final["submission_path"]},
        ]
    )
    display(tft_final_meta)
    display(tft_final["prediction_summary"])
    display(tft_final["submission_head"])
except ImportError as exc:
    print(exc)
except FileNotFoundError as exc:
    print(exc)


## Выводы после EDA и baselines

- Временной ряд явно нестационарен: видны долгосрочный рост, сильная недельная сезонность и отдельные режимные сдвиги.
- Выходные заметно сильнее будней; дни 15 числа и конца месяца тоже выглядят выше среднего, что согласуется с описанием задачи.
- `transactions` хорошо сонаправлены с продажами, поэтому это сильный кандидат в baseline-признаки.
- `onpromotion` нельзя бездумно превращать в `False`: доля пропусков существенная, а сами нули продаж в train отсутствуют.
- Землетрясение 16 апреля 2016 года создаёт локальный structural break, который стоит учитывать отдельным признаком/режимом.
- В `test.csv` есть unseen items, значит для cold-start по item нужен backoff хотя бы до `family`/`class`.
- Для holidays обязательно нужно отдельно обработать `transferred`, `Bridge` и `Work Day`, иначе календарные признаки будут шумными.
- Из двух простых baseline-моделей лучше работает иерархическая схема с `weekday`/`family`/`promotion`.
- Первый `LightGBM` с табличными признаками уже заметно улучшает holdout-метрику относительно lookup-baseline.
- Более строгий rolling-origin CV показал, что окно `224` дня стабильнее и лучше `112` по среднему WRMSLE.
- После первого Kaggle прогона стало ясно, что observed-row holdout слишком оптимистичен, поэтому основной акцент сместился на pseudo-public full-panel validation.
- Текущая боевая модель - `CatBoost` на sampled recent train с aggregate priors и recent-train cache.
- Лучший вручную проверенный `CatBoost` fold сейчас заметно ближе к реальной leaderboard-логике, чем старые observed-row offline scores.
- Главные драйверы текущей модели: `si_recent28`, `siw_mean`, `siw_count_log`, `onpromotion` и соотношение recent `28/56` дней.

Что можно делать следующим шагом:
1. добить полный `CatBoost` rolling CV для `lookback_grid=(168, 224)`;
2. запустить более длинный `Optuna`, а не мини-поиск на `2` trial'ах;
3. отдельно валидировать cold-start fallback и только потом возвращать его в final blend.